## **Finland labor market gig shift 2019-2026**

In [2]:
import numpy as np
import pandas as pd

## 1. Traficom Taxi Licences (2016–2023)
Source: Finnish Transport and Communications Agency (Traficom)
Issues to fix:
- Semicolon separator instead of comma
- Thousands separator spaces in licence counts e.g. "9 624" → 9624
- Date column needs splitting into year and month

In [ ]:
# Load raw file 
df_taxi = pd.read_csv('../data/raw/traficom_taxi_licences_2016_2023.csv', sep=';')

In [4]:
df_taxi.shape

(9, 2)

In [5]:
df_taxi.head()

,Date,Number of licenses
0,12/2016,9 624
1,12/2017,9 499
2,6/2018,9 487
3,12/2018,11 852
4,12/2019,12 332


In [6]:
df_taxi.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Date                9 non-null      str  
 1   Number of licenses  9 non-null      str  
dtypes: str(2)
memory usage: 276.0 bytes


In [7]:
df_taxi.isnull().sum()

Date                  0
Number of licenses    0
dtype: int64

In [8]:
# Fix column names to lowercase with underscores
df_taxi.columns = ['date', 'number_of_licences']

In [9]:
# Convert date to datetime
df_taxi['date'] = pd.to_datetime(df_taxi['date'], format='%m/%Y')

In [10]:
df_taxi['date'].dtype

dtype('<M8[us]')

In [11]:
# Remove thousands separator spaces and convert to integer
df_taxi['number_of_licences'] = df_taxi['number_of_licences'].str.replace(' ', '').astype(int)

In [12]:
# Extract year and month as separate columns
df_taxi['year'] =  df_taxi['date'].dt.year
df_taxi['month']= df_taxi['date'].dt.month

In [13]:
# Add source column for traceability in PostgreSQL later
df_taxi['source'] = 'traficom'

In [14]:
df_taxi.head()

,date,number_of_licences,year,month,source
0,2016-12-01,9624,2016,12,traficom
1,2017-12-01,9499,2017,12,traficom
2,2018-06-01,9487,2018,6,traficom
3,2018-12-01,11852,2018,12,traficom
4,2019-12-01,12332,2019,12,traficom


In [15]:
df_taxi.dtypes

date                  datetime64[us]
number_of_licences             int64
year                           int32
month                          int32
source                           str
dtype: object

In [ ]:
# Export cleaned dataframe to CSV for PostgreSQL import

df_taxi.to_csv('../data/cleaned/traficom_taxi_licences_2016_2023.csv', index=False)
print("Exported Succesfully")

Exported Succesfully


## 2. StatFin Population by Activity Type (1987–2024)
Source: Statistics Finland (stat.fi)
Issues to fix:
- Wide format (years as columns) needs reshaping to long format (one row per year)
- First row is a header artifact from stat.fi export
- Values are percentages of total population

In [ ]:
# Load raw file without headers since first rows contain metadata, not column names

df_population = pd.read_csv('../data/raw/statfin_population_by_activity_1987_2024.csv', header=None)

In [28]:
df_population

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,Population by main type of activity by municip...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Whole country,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1987.0,1988.0,1989.0,1990.0,1991.0,1992.0,1993.0,1994.0,1995.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0
3,Employed,47.0,47.5,47.7,46.7,43.1,39.9,37.0,37.6,37.8,...,41.1,41.4,42.2,43.0,43.0,41.3,42.8,43.6,43.1,43.0
4,Unemployed,3.0,2.6,2.2,2.8,6.0,8.8,10.5,9.6,9.3,...,6.8,6.5,5.4,4.6,4.7,6.4,4.9,4.6,5.3,5.7
5,0-14 years old,19.3,19.4,19.3,19.3,19.2,19.2,19.1,19.1,19.0,...,16.3,16.2,16.2,16.0,15.8,15.6,15.4,15.1,14.9,14.6
6,"Students, pupils",6.3,6.2,6.4,6.6,7.1,7.4,8.4,8.7,8.9,...,7.5,7.4,7.3,7.1,7.3,7.6,7.6,7.4,7.6,7.9
7,"Conscripts, persons in non-military service",0.6,0.5,0.4,0.6,0.4,0.3,0.4,0.4,0.3,...,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
8,Pensioners,20.3,20.4,20.6,20.8,20.9,21.0,21.3,21.5,21.6,...,24.9,25.3,25.7,25.9,26.0,26.0,26.0,26.0,25.8,25.6
9,Other persons outside the labour force,3.6,3.4,3.3,3.3,3.2,3.4,3.3,3.2,3.2,...,3.1,3.1,3.2,3.2,3.2,3.1,3.1,3.2,3.3,3.1


In [18]:
df_population.shape

(10, 39)

In [19]:
df_population.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 39 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Whole country  9 non-null      str    
 1   Unnamed: 1     8 non-null      float64
 2   Unnamed: 2     8 non-null      float64
 3   Unnamed: 3     8 non-null      float64
 4   Unnamed: 4     8 non-null      float64
 5   Unnamed: 5     8 non-null      float64
 6   Unnamed: 6     8 non-null      float64
 7   Unnamed: 7     8 non-null      float64
 8   Unnamed: 8     8 non-null      float64
 9   Unnamed: 9     8 non-null      float64
 10  Unnamed: 10    8 non-null      float64
 11  Unnamed: 11    8 non-null      float64
 12  Unnamed: 12    8 non-null      float64
 13  Unnamed: 13    8 non-null      float64
 14  Unnamed: 14    8 non-null      float64
 15  Unnamed: 15    8 non-null      float64
 16  Unnamed: 16    8 non-null      float64
 17  Unnamed: 17    8 non-null      float64
 18  Unnamed: 18    8 non-nul

In [26]:
df_population.head()

,Whole country,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38
0,NaN,1987.0,1988.0,1989.0,1990.0,1991.0,1992.0,1993.0,1994.0,1995.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0
1,Employed,47.0,47.5,47.7,46.7,43.1,39.9,37.0,37.6,37.8,...,41.1,41.4,42.2,43.0,43.0,41.3,42.8,43.6,43.1,43.0
2,Unemployed,3.0,2.6,2.2,2.8,6.0,8.8,10.5,9.6,9.3,...,6.8,6.5,5.4,4.6,4.7,6.4,4.9,4.6,5.3,5.7
3,0-14 years old,19.3,19.4,19.3,19.3,19.2,19.2,19.1,19.1,19.0,...,16.3,16.2,16.2,16.0,15.8,15.6,15.4,15.1,14.9,14.6
4,"Students, pupils",6.3,6.2,6.4,6.6,7.1,7.4,8.4,8.7,8.9,...,7.5,7.4,7.3,7.1,7.3,7.6,7.6,7.4,7.6,7.9


In [30]:
# Extract years from row 2 (skip first cell which is NaN)
years = df_population.iloc[2, 1:].tolist()
years = [int(float(y)) for y in years]

In [31]:
# Extract categories and values from rows 3 to 9
categories = df_population.iloc[3:10, 0].tolist()
values = df_population.iloc[3:10, 1:].values

In [32]:
# Build clean long format dataframe
rows = []
for i, category in enumerate(categories):
    for j, year in enumerate(years):
        rows.append({
            'year': year,
            'category': category,
            'value_pct': values[i][j],
            'source': 'stat.fi'
        })

df_population = pd.DataFrame(rows)

In [33]:
    # Filter to 2019-2024
df_population = df_population[df_population['year'] >= 2019]

In [ ]:
df_population.head()

,year,category,value_pct,source
32,2019,Employed,43.0,stat.fi
33,2020,Employed,41.3,stat.fi
34,2021,Employed,42.8,stat.fi
35,2022,Employed,43.6,stat.fi
36,2023,Employed,43.1,stat.fi


In [ ]:
# Export cleaned dataframe to CSV for PostgreSQL import

df_population.to_csv('../data/cleaned/statfin_population_by_activity_1987_2024.csv', index=False)

## 3. StatFin Employment Rate Monthly (2010–2026)
Source: Statistics Finland (stat.fi)
Issues to fix:
- No proper column names — first row contains month labels, not headers
- Date format is '2010M01' — needs splitting into separate year and month columns
- Rows 3 and 4 are junk (Unit and Source footnotes) — need to be dropped
- Wide format — months spread across 196 columns, needs reshaping to one row per month

In [ ]:
# Load raw file without headers since first rows contain metadata, not column names

df_employment = pd.read_csv('../data/raw/statfin_employment_rate_monthly_2010_2026.csv',header=None)

In [42]:
df_employment.head()

,Employment rate and trend of employment rate of persons aged 20 to 64 in 2010M01 to 2026M04,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 187,Unnamed: 188,Unnamed: 189,Unnamed: 190,Unnamed: 191,Unnamed: 192,Unnamed: 193,Unnamed: 194,Unnamed: 195,Unnamed: 196
0,NaN,2010M01,2010M02,2010M03,2010M04,2010M05,2010M06,2010M07,2010M08,2010M09,...,2025M07,2025M08,2025M09,2025M10,2025M11,2025M12,2026M01,2026M02,2026M03,2026M04
1,Employment rate,69.5,70.7,70.8,70.8,72.9,73.1,73.1,73.4,71.4,...,76.5,76.2,75.9,76.3,75.6,75.0,74.8,75.3,74.2,74.7
2,"Employment rate, trend",71.6,71.6,71.6,71.6,71.7,71.7,71.7,71.8,71.8,...,75.8,75.9,75.9,75.9,75.9,75.7,75.7,75.6,75.4,75.3
3,Unit: %,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Source: Statistics Finland, labour force survey",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
## Extract row 0 to column header
df_employment.columns = ['category'] + df_employment.iloc[0, 1:].tolist()

## Drop  row 0 since it's now a header
df_employment = df_employment.drop(index=0)

In [46]:
# Drop junk rows 3 and 4
df_employment =df_employment.drop(index=[3,4])

In [ ]:
# Reshape from wide to long format
# Each month column becomes a row, creating 'month_code' and 'rate' columns

df_employment = df_employment.melt(id_vars='category',
                                   var_name='month_code',
                                   value_name='rate')

In [ ]:
# Split '2010M01' into separate year and month integer columns
df_employment['year'] = df_employment['month_code'].str[:4].astype(int)
df_employment['month'] = df_employment['month_code'].str[5:].astype(int)

# Drop month_code as it's no longer needed — year and month columns replace it
df_employment = df_employment.drop(columns='month_code')

In [65]:
# Convert rate to float
df_employment['rate'] = df_employment['rate'].astype(float)

In [66]:
# Filter to 2019 onwards
df_employment =df_employment[df_employment['year'] >= 2019]

In [ ]:
## # Add source column
df_employment['source'] = 'stat.fi'

In [68]:
df_employment.head()

,category,rate,year,month,source
216,Employment rate,74.9,2019,1,stat.fi
217,"Employment rate, trend",75.8,2019,1,stat.fi
218,Employment rate,74.3,2019,2,stat.fi
219,"Employment rate, trend",75.8,2019,2,stat.fi
220,Employment rate,75.6,2019,3,stat.fi


In [69]:
df_employment.dtypes

category        str
rate        float64
year          int64
month         int64
source          str
dtype: object

In [ ]:
# Export cleaned dataframe to CSV for PostgreSQL import

df_employment.to_csv('../data/cleaned/statfin_employment_rate_monthly_2010_2026.csv', index=False)

In [23]:
df_eurostat=pd.read_csv('../data/raw/eurostat_unemployment_monthly_eu_2025.csv')

In [24]:
df_trends_yrittajaksi=pd.read_csv('../data/raw/google_trends_yrittajaksi_2019_2026.csv')

In [25]:
df_trends_keikkayto=pd.read_csv('../data/raw/google_trends_keikkayto_2019_2026.csv')